# Placental Vessel, IVS, and Villi Quantification



In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys, os
# Add this notebook's folder to sys.path so `import iuquant` works. Jupyter has
# no __file__, so fall back to the current working directory.
nb_dir = os.getcwd()
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)


import re
import tifffile  
import iuquant
from iuquant import (
    list_segmentations, load_and_crop, analyze_volume_fractions,
    QuantSettings, PreprocessSettings, make_measurements, save_metrics_csv,
    plot_metric_summary, view_maps,
)

# A. Vessel Batch

## 1. List the Vessel TIFFs

In [ ]:
imgs_path = "/path/to/quantification-notebooks/vessel/"   # <--  Copy `vessel` folder path from current directory
names = list_segmentations(imgs_path, pattern="*.tiff")  # [(basename, fullpath), ...]
print(len(names), "files")
names

## 2. Full quantification 


In [ ]:
# Optional preprocessing of the full-res binary mask, before downsampling and
# any measurement. Set enabled=True to use. Sizes/radii are in fulll-res voxels.
preprocess = PreprocessSettings(
    enabled=True,                  # <-- flip to True to apply the cleanup below
    median_size=2,                  # cubic median (majority) filter; 0/1 = off
    morphology=[("open", 1),        # remove small protrusions/specks
                ("close", 1)],      # close = bridge gaps; ops run in this order
    fill_hole_size=64,              # fill enclosed holes up to this many voxels (0 = off)
    min_object_size=64,             # drop connected components < this many voxels
    max_object_size=None,           # or drop very large blobs (None = off)
    connectivity=3,                 # 1/2/3 -> 6/18/26-neighbour
)

settings = QuantSettings(
    local_thickness=True,
    skeletonization=True,
    surface_area=True,
    pore_network=False,
    ds=2,                                      # downsample factor
    conv_factor=0.8125,                        # micrometres per voxel
    crop=[0, 900, 0, 900, 0, 900],       # [z0,z1,y0,y1,x0,x1] or None
    calc_ivs=False,                            # True -> intervillous space
    multilabel=False,                          # True -> one analysis per label
    preprocess=preprocess,                     # mask cleanup (above)
    save_maps=True,                            # persist skeletons/thickness/crops
    upscale_maps=True,                         # maps in r.maps -> full-res, align in napari
    from_scratch=True,                         # False -> reload saved maps
    output_dir="./out",
   
    teasar_params={
        "scale": 2,    # invalidation radius = scale*local_radius + const
        "const": 50,  
        # "max_paths": None,             # cap number of branches (None = unlimited)
        # "pdrf_exponent": 4,            # higher -> paths hug the centreline more
        # "soma_detection_threshold": 0,# 0 disables soma handling (good for tubes)
    },
    dust_threshold=1000,  # skeleton components smaller than 1000 voxels (kimimaro clean up) will be removed
    # skeletonize_kwargs={"anisotropy": (2.0, 1.0, 1.0)},  # extra kimimaro.skeletonize opts
)
vessel_results = make_measurements(names, settings)    # keep a handle for the combined napari view below

## 3. Combined metrics CSV

In [ ]:
# Save out metrics
out_csv = "vessel_metrics.csv"
df = save_metrics_csv([r.metrics for r in vessel_results], out_csv)
df

In [ ]:
# Split the filename column and extract the scan number part
df['filename'] = df['filename'].str.split('_', expand=True)[1]

In [ ]:
# plot metric summary
plot_metric_summary(df, columns=None, kind="bar", output_folder=settings.output_dir);

## 4. Volume and area fraction analysis 

Random-crop volume fraction vs ROI size, area fraction per slice, and a slice
montage — run **after** quantification, on the same preprocessed masks
(`r.mask`, binary 0/1, so `fg_idx=1`). To analyse the raw files on disk instead,
pass `names` here and set `fg_idx=255`.

In [ ]:
# Feed the preprocessed masks from quantification 
vf_inputs = [(r.image, r.mask) for r in vessel_results if r.mask is not None]

vf_results = analyze_volume_fractions(
    vf_inputs,
    output_folder=None,                       # e.g. "/ceph/.../VFAF_out" to save
    roi_sizes=[32, 64, 96, 128, 160, 192],    # increasing ROI sides (voxels)
    img_slices=[32, 64, 96, 128, 160, 192],
    n_crops=10,
    subvol_size=256,                          # representative sub-volume side
    seed=0,                                   # reproducible random crops
    plot_titles=None,
    fg_idx=1,                                 # preprocessed masks are 0/1
)

# B. IVS Batch

## 1. List the IVS TIFFs

In [ ]:
imgs_path = "/path/to/quantification-notebooks/villi/"   # <--  Copy `villi` folder path from current directory
names = list_segmentations(imgs_path, pattern="*.tiff")                              # [(basename, fullpath), ...]
print(len(names), "files")
names

## 2. Full quantification 


In [ ]:
# Optional preprocessing of the full-res binary mask, before downsampling and
# any measurement. Set enabled=True to use. Sizes/radii are in full-res voxels.

preprocess = PreprocessSettings(
    enabled=False,                  # <-- flip to True to apply the cleanup below
    median_size=2,                  # cubic median (majority) filter; 0/1 = off
    morphology=[("open", 1),        # remove small protrusions/specks
                ("close", 1)],      # close = bridge gaps; ops run in this order
    fill_hole_size=64,              # fill enclosed holes up to this many voxels (0 = off)
    min_object_size=64,             # drop connected components < this many voxels
    max_object_size=None,           # or drop very large blobs (None = off)
    connectivity=3,                 # 1/2/3 -> 6/18/26-neighbour
)

settings = QuantSettings(
    local_thickness=True,
    skeletonization=False,
    surface_area=True,
    pore_network=True,
    ds=2,                                      # downsample factor
    conv_factor=0.8125,                        # micrometres per voxel
    crop=[0, 900, 0, 900, 0, 900],       # [z0,z1,y0,y1,x0,x1] or None
    calc_ivs=True,                            # True -> intervillous space
    multilabel=False,                          # True -> one analysis per label
    preprocess=preprocess,                     # mask cleanup (above)
    save_maps=True,                            # persist skeletons/thickness/crops
    upscale_maps=True,                         # maps in r.maps -> full-res, align in napari
    from_scratch=True,                         # False -> reload saved maps
    output_dir="./out",
   
    teasar_params={
        "scale": 2,    # invalidation radius = scale*local_radius + const
        "const": 50,   # lower scale/const -> more, finer branches; higher -> pruned
        # "max_paths": None,             # cap number of branches (None = unlimited)
        # "pdrf_exponent": 4,            # higher -> paths hug the centreline more
        # "soma_detection_threshold": 0,# 0 disables soma handling (good for tubes)
    },
    dust_threshold=1000,   # only drops disconnected blobs < N vox (no effect on one solid network)
    # skeletonize_kwargs={"anisotropy": (2.0, 1.0, 1.0)},  # extra kimimaro.skeletonize opts
)
ivs_results = make_measurements(names, settings)      # keep a handle for the combined napari view at the end

## 3. Combined metrics CSV

In [ ]:
# save out metrics
out_csv = "ivs_metrics.csv"
df = save_metrics_csv([r.metrics for r in ivs_results], out_csv)
df

In [ ]:
# Split the filename column and extract the scan number part
df['filename'] = df['filename'].str.split('_', expand=True)[1]

In [ ]:
plot_metric_summary(df, columns=None, kind="bar", output_folder=settings.output_dir);

## 4. Volume and area fraction analysis 

Random-crop volume fraction vs ROI size, area fraction per slice, and a slice
montage — run **after** quantification, on the same preprocessed masks
(`r.mask`, binary 0/1, so `fg_idx=1`). To analyse the raw files on disk instead,
pass `names` here and set `fg_idx=255`.

In [ ]:
# Feed the preprocessed masks from quantification 
vf_inputs = [(r.image, r.mask) for r in ivs_results if r.mask is not None]

vf_results = analyze_volume_fractions(
    vf_inputs,
    output_folder=None,                       # e.g. "/ceph/.../VFAF_out" to save
    roi_sizes=[32, 64, 96, 128, 160, 192],    # increasing ROI sides (voxels)
    img_slices=[32, 64, 96, 128, 160, 192],
    n_crops=10,
    subvol_size=256,                          # representative sub-volume side
    seed=0,                                   # reproducible random crops
    plot_titles=None,
    fg_idx=1,                                 # preprocessed masks are 0/1
)

# C. Villi Batch

## 1. List the Villi TIFFs

In [ ]:
imgs_path = "/path/to/quantification-notebooks/villi/"   # <--  Copy `villi` folder path from current directory
names = list_segmentations(imgs_path, pattern="*.tiff")                              # [(basename, fullpath), ...]
print(len(names), "files")
names

## 2. Full quantification 


In [ ]:
# Optional preprocessing of the full-res binary mask, before downsampling and
# any measurement. Set enabled=True to use. Sizes/radii are in full-res voxels.

preprocess = PreprocessSettings(
    enabled=False,                  # <-- flip to True to apply the cleanup below
    median_size=2,                  # cubic median (majority) filter; 0/1 = off
    morphology=[("open", 1),        # remove small protrusions/specks
                ("close", 1)],      # close = bridge gaps; ops run in this order
    fill_hole_size=64,              # fill enclosed holes up to this many voxels (0 = off)
    min_object_size=64,             # drop connected components < this many voxels
    max_object_size=None,           # or drop very large blobs (None = off)
    connectivity=3,                 # 1/2/3 -> 6/18/26-neighbour
)

settings = QuantSettings(
    local_thickness=True,
    skeletonization=False,
    surface_area=True,
    pore_network=False,
    ds=2,                                      # downsample factor
    conv_factor=0.8125,                        # micrometres per voxel
    crop=[0, 900, 0, 900, 0, 900],       # [z0,z1,y0,y1,x0,x1] or None
    calc_ivs=False,                            # True -> intervillous space
    multilabel=False,                          # True -> one analysis per label
    preprocess=preprocess,                     # mask cleanup (above)
    save_maps=True,                            # persist skeletons/thickness/crops
    upscale_maps=True,                         # maps in r.maps -> full-res, align in napari
    from_scratch=True,                         # False -> reload saved maps
    output_dir="./out",
   
    teasar_params={
        "scale": 2,    # invalidation radius = scale*local_radius + const
        "const": 50,   # lower scale/const -> more, finer branches; higher -> pruned
        # "max_paths": None,             # cap number of branches (None = unlimited)
        # "pdrf_exponent": 4,            # higher -> paths hug the centreline more
        # "soma_detection_threshold": 0,# 0 disables soma handling (good for tubes)
    },
    dust_threshold=1000,   # only drops disconnected blobs < N vox (no effect on one solid network)
    # skeletonize_kwargs={"anisotropy": (2.0, 1.0, 1.0)},  # extra kimimaro.skeletonize opts
)
results = make_measurements(names, settings)
villi_results = results        # keep a handle for the combined napari view at the end

## 3. Combined metrics CSV

In [ ]:
# save out metrics
out_csv = "villi_metrics.csv"
df = save_metrics_csv([r.metrics for r in villi_results], out_csv)
df

In [ ]:
# Split the filename column and extract the scan number part
df['filename'] = df['filename'].str.split('_', expand=True)[1]

In [ ]:
plot_metric_summary(df, columns=None, kind="bar", output_folder=settings.output_dir);

## 4. Volume and area fraction analysis 

Random-crop volume fraction vs ROI size, area fraction per slice, and a slice
montage — run **after** quantification, on the same preprocessed masks
(`r.mask`, binary 0/1, so `fg_idx=1`). To analyse the raw files on disk instead,
pass `names` here and set `fg_idx=255`.

In [ ]:
# Feed the preprocessed masks from quantification 
vf_inputs = [(r.image, r.mask) for r in villi_results if r.mask is not None]

vf_results = analyze_volume_fractions(
    vf_inputs,
    output_folder=None,                       # e.g. "/ceph/.../VFAF_out" to save
    roi_sizes=[32, 64, 96, 128, 160, 192],    # increasing ROI sides (voxels)
    img_slices=[32, 64, 96, 128, 160, 192],
    n_crops=10,
    subvol_size=256,                          # representative sub-volume side
    seed=0,                                   # reproducible random crops
    plot_titles=None,
    fg_idx=1,                                 # preprocessed masks are 0/1
)

In [ ]:
out_dir = "/path/to/quantification-notebooks/out" # Copy `out` folder path from current directory

# each scan:
scans = [
    {"sample_id": "123531", "image_path": "/path/to/quantification-notebooks/sample_123531_IMAGE.tiff"},
    # {"sample_id": "123642", "image_path": "/path/to/quantification-notebooks/sample_123642_IMAGE.tiff"},
    # {"sample_id": "123761", "image_path": "/path/to/quantification-notebooks/sample_123761_IMAGE.tiff"},
]


In [ ]:
from scipy.ndimage import zoom  

# Downsample an array by a factor of ds using local mean. This is slower but produces smoother results.
def downsample_mean(arr, ds):
    """Slower but smoother: averages ds x ds x ds blocks."""
    from skimage.transform import downscale_local_mean
    return downscale_local_mean(arr, (ds, ds, ds)).astype(arr.dtype)

In [ ]:
pattern = re.compile(r"sample_(\d+)_(VESSEL|VILLI)_(.+)\.tiff?$", re.IGNORECASE) # match sample_id, tag, metric

### Load each scan's image and all output files for that sample_id, then view in napari

In [ ]:
layers = {}
for scan in scans:
    sample_id = scan["sample_id"]

    # Load this scan's raw image first
    if os.path.exists(scan["image_path"]):
        layers[f"{sample_id}_image"] = load_and_crop(scan["image_path"], settings.crop)
        layers[f"{sample_id}_image_ds{settings.ds}"] = downsample_mean(layers[f"{sample_id}_image"], settings.ds)
        print(f"{sample_id}: loaded image {layers[f'{sample_id}_image'].shape} + downsampled {layers[f'{sample_id}_image_ds{settings.ds}'].shape}")
    else:
        print(f"{sample_id}: image not found at {scan['image_path']}")

    # Load every output file for this sample_id before moving to the next scan
    matched = 0
    for fname in sorted(os.listdir(out_dir)):
        m = pattern.match(fname)
        if not m:
            continue
        f_sample_id, tag, metric = m.groups()
        if f_sample_id != sample_id:
            continue  # belongs to a different scan, skip for now

        fpath = os.path.join(out_dir, fname)
        try:
            arr = tifffile.imread(fpath)
        except Exception as e:
            print(f"  skipped {fname}: {e}")
            continue

        key = f"{sample_id}_{tag.lower()}_{metric}"
        layers[key] = arr
        matched += 1

    print(f"{sample_id}: loaded {matched} output file(s) for sample {sample_id}")

view_maps(layers)